In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import joblib
import os

In [3]:
df = pd.read_csv(
    r'C:\Users\boser\Desktop\Project_Carbon\data\processed\final_dataset_engineered.csv',
    index_col='date',
    parse_dates=True
)

TARGET = 'carbon_price_change_pct'

FEATURES = [
    'gas_price_weekly', 'gas_price_lag1', 'gas_price_volatility',
    'vix', 'co2_change', 'days_to_next_cop', 'cop_urgency',
    'is_cop_week', 'recent_volatility', 'momentum_4w',
    'momentum_12w', 'regime'
]

X = df[FEATURES]
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (577, 12)
y shape: (577,)


In [5]:
split_index = int(len(df) * 0.80)
split_date  = df.index[split_index]

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

print(f"Split date          : {split_date.date()}")
print(f"Training rows       : {len(X_train)} weeks")
print(f"Test rows           : {len(X_test)} weeks")
print(f"Training period     : {X_train.index.min().date()} → {X_train.index.max().date()}")
print(f"Test period         : {X_test.index.min().date()} → {X_test.index.max().date()}")

Split date          : 2024-02-04
Training rows       : 461 weeks
Test rows           : 116 weeks
Training period     : 2015-04-05 → 2024-01-28
Test period         : 2024-02-04 → 2026-04-26


In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # Fit + transform train
X_test_scaled  = scaler.transform(X_test)         # Transform only — no fit

X_train_scaled = pd.DataFrame(X_train_scaled, 
                            columns=FEATURES, 
                            index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  
                            columns=FEATURES, 
                            index=X_test.index)

print("Scaling complete")
print("\nX_train_scaled stats (should be near mean=0, std=1):")
print(X_train_scaled.describe().loc[['mean','std']].round(3))

Scaling complete

X_train_scaled stats (should be near mean=0, std=1):
      gas_price_weekly  gas_price_lag1  gas_price_volatility    vix  \
mean             0.000           0.000                -0.000 -0.000   
std              1.001           1.001                 1.001  1.001   

      co2_change  days_to_next_cop  cop_urgency  is_cop_week  \
mean       0.000            -0.000        0.000       -0.000   
std        1.001             1.001        1.001        1.001   

      recent_volatility  momentum_4w  momentum_12w  regime  
mean             -0.000       -0.000         0.000   0.000  
std               1.001        1.001         1.001   1.001  


In [7]:
tscv = TimeSeriesSplit(n_splits=5)

print("Time Series Cross Validation — 5 Splits")
print("=" * 50)
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_scaled)):
    train_dates = X_train_scaled.index[train_idx]
    val_dates   = X_train_scaled.index[val_idx]
    print(f"Fold {fold+1}: Train {train_dates.min().date()} → {train_dates.max().date()} "
        f"| Val {val_dates.min().date()} → {val_dates.max().date()}")

Time Series Cross Validation — 5 Splits
Fold 1: Train 2015-04-05 → 2016-10-16 | Val 2016-10-23 → 2018-04-01
Fold 2: Train 2015-04-05 → 2018-04-01 | Val 2018-04-08 → 2019-09-15
Fold 3: Train 2015-04-05 → 2019-09-15 | Val 2019-09-22 → 2021-02-28
Fold 4: Train 2015-04-05 → 2021-02-28 | Val 2021-03-07 → 2022-08-14
Fold 5: Train 2015-04-05 → 2022-08-14 | Val 2022-08-21 → 2024-01-28


In [8]:
models_path = r'C:\Users\boser\Desktop\Project_Carbon\models'
os.makedirs(models_path, exist_ok=True)

joblib.dump(scaler, os.path.join(models_path, 'scaler.pkl'))

# Save splits for use in training notebook
X_train_scaled.to_csv(os.path.join(models_path, 'X_train.csv'))
X_test_scaled.to_csv(os.path.join(models_path,  'X_test.csv'))
y_train.to_csv(os.path.join(models_path, 'y_train.csv'))
y_test.to_csv(os.path.join(models_path,  'y_test.csv'))

print("Saved:")
print(f"  scaler.pkl — {os.path.join(models_path, 'scaler.pkl')}")
print(f"  X_train.csv, X_test.csv, y_train.csv, y_test.csv")

Saved:
  scaler.pkl — C:\Users\boser\Desktop\Project_Carbon\models\scaler.pkl
  X_train.csv, X_test.csv, y_train.csv, y_test.csv
